# 🔍 Modul 03: Exploratory Data Analysis (EDA) & Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/03_EDA_and_Feature_Engineering.ipynb)

> **Repositori Resmi**: [github.com/antonprafanto/data-mining-zero-to-hero](https://github.com/antonprafanto/data-mining-zero-to-hero)  
> **Tingkat Kesulitan**: Pemula Menengah (From Zero to Hero)  
> **Estimasi Waktu Belajar**: 60 – 75 Menit (Santai dan interaktif)  
> **Tujuan Modul**: Menguasai alur kerja EDA terstruktur (Univariat, Bivariat, Multivariat via Pairplot & Heatmap), Seni Rekayasa Fitur (*Feature Engineering*: ekstraksi datetime, diskretisasi `pd.cut` & `pd.qcut`, fitur rasio), memahami bahaya *Curse of Dimensionality*, serta seleksi fitur cerdas (*Feature Importance*).

---

## 💡 1. Filosofi: Seni Menggali Cerita & Merancang Fitur

> *"Applied Machine Learning is basically Feature Engineering."* — **Prof. Andrew Ng (Stanford University)**

Banyak orang awam mengira bahwa keberhasilan model data mining ditentukan oleh algoritma yang paling rumit atau nama yang paling keren.  
**Faktanya:**  
- Model machine learning sederhana (seperti Decision Tree atau Regresi Linier) jika disuapi dengan **fitur-fitur baru yang cerdas dan kaya informasi**, akan dengan mudah mengalahkan model deep learning rumit yang hanya diberi fitur mentah!
- **Exploratory Data Analysis (EDA)** adalah proses detektif untuk memahami data kita secara visual sebelum memutuskan model apa yang akan dibuat.

Mari kita pelajari alur kerja EDA terstruktur dan seni rekayasa fitur langkah demi langkah!

---

## 📊 2. Alur EDA Bagian 1: Analisis Univariat (Satu per Satu Variabel)

Analisis Univariat bertujuan memahami **karakteristik dan distribusi dari setiap kolom secara terpisah**:
- Untuk angka numerik: Kita ukur kemiringannya (**Skewness**) dan keruncingannya (**Kurtosis**).
- Untuk kategori teks: Kita ukur frekuensi kemunculannya menggunakan **Diagram Batang (*Countplot*)**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Memuat dataset riil transaksi restoran (Tips Dataset)
df = sns.load_dataset('tips')
print(f"Ukuran Dataset: {df.shape[0]} baris × {df.shape[1]} kolom")
df.head()

In [ ]:
# Analisis Distribusi Kolom Numerik (Total Tagihan & Tip)
print("=== UKURAN BENTUK DISTRIBUSI (SKEWNESS & KURTOSIS) ===")
print(f"Skewness Total Bill : {df['total_bill'].skew():.2f} (Positif = Miring ke Kanan/Right-Skewed)")
print(f"Kurtosis Total Bill : {df['total_bill'].kurt():.2f}")

# Visualisasi Histogram Distribusi + KDE
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
sns.histplot(df['total_bill'], kde=True, color='teal')
plt.title('Distribusi Total Tagihan ($)', fontweight='bold')

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='day', hue='day', palette='Set2', legend=False)
plt.title('Frekuensi Transaksi per Hari', fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📈 3. Alur EDA Bagian 2: Analisis Bivariat (Hubungan Antar 2 Variabel)

Di tahap ini, kita menyelidiki: *"Apakah variabel A mempengaruhi variabel B?"*

1. **Korelasi Pearson vs Spearman**:
   - **Pearson**: Mengukur hubungan garis lurus (*linear*).
   - **Spearman**: Mengukur hubungan berurutan peringkat (*monotonic rank*), sangat bagus jika data memiliki outlier.
2. **Tabulasi Silang (*Cross-tabulation*)**: Mengukur perpaduan dua variabel kategori menggunakan `pd.crosstab()`.
3. **Boxplot Komparatif**: Membandingkan angka numerik di setiap kelompok kategori.

In [ ]:
# 1. Mengukur Korelasi antara Total Tagihan dan Besar Tip yang Diberikan
corr_pearson = df['total_bill'].corr(df['tip'], method='pearson')
corr_spearman = df['total_bill'].corr(df['tip'], method='spearman')
print(f"Korelasi Pearson  (Linear)    : {corr_pearson:.3f}")
print(f"Korelasi Spearman (Peringkat) : {corr_spearman:.3f}")
print("Kesimpulan: Ada korelasi positif kuat! Semakin mahal tagihan, semakin besar tip yang diberikan.")

# 2. Tabulasi Silang Kategori: Waktu Makan (Lunch/Dinner) vs Hari
print("\n=== TABEL SILANG (CROSS-TABULATION): HARI VS WAKTU MAKAN ===")
tabel_silang = pd.crosstab(df['day'], df['time'], margins=True)
print(tabel_silang)

In [ ]:
# Visualisasi Komparatif: Total Tagihan Berdasarkan Hari & Jenis Kelamin
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='day', y='total_bill', hue='sex', palette='Pastel1')
plt.title('Perbandingan Total Tagihan per Hari Berdasarkan Jenis Kelamin', fontweight='bold')
plt.ylabel('Total Tagihan ($)')
plt.show()

---

## 🌐 4. Alur EDA Bagian 3: Analisis Multivariat (Melihat Seluruh Variabel Serentak)

1. **Pairplot (`sns.pairplot`)**: Diagram sakti paling populer dalam data mining. Menampilkan matriks scatter plot antar seluruh pasangan kolom angka sekaligus diwarnai oleh target kategori!
2. **Heatmap Korelasi**: Memetakan angka korelasi (-1.0 s/d +1.0) dalam kanvas warna.

In [ ]:
# 1. Pairplot Matriks Sebaran Multivariat
sns.pairplot(df, hue='time', palette='Dark2', diag_kind='kde')
plt.suptitle('Pairplot Multivariat Dataset Tips Restoran', y=1.02, fontweight='bold')
plt.show()

In [ ]:
# 2. Heatmap Korelasi Numerik
plt.figure(figsize=(6, 4))
kolom_numerik = df.select_dtypes(include=[np.number])
sns.heatmap(kolom_numerik.corr(), annot=True, cmap='Blues', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriks Korelasi Numerik', fontweight='bold')
plt.show()

---

## 🛠️ 5. Seni Rekayasa Fitur (*Feature Engineering*)

Sekarang kita masuki keahlian paling berharga bagi praktisi data: **menciptakan fitur baru dari data yang ada**!

### A. Teknik 1: Ekstraksi Fitur Tanggal & Waktu (`datetime`)
Data tanggal seperti `"2026-10-06 19:45:00"` tidak bisa dicerna langsung oleh model. Tapi jika kita pecah menjadi:
- Jam transaksi (Pagi, Siang, Malam)
- Hari kerja (*Weekday*) vs Akhir pekan (*Weekend*)
- Kuartal (Q1, Q2, Q3, Q4)

Maka model machine learning akan langsung mengenali pola kebiasaan pelanggan!

In [ ]:
# Simulasi data waktu transaksi restoran
waktu_simulasi = pd.date_range(start='2026-10-01 10:00:00', periods=len(df), freq='3h')
df['waktu_transaksi'] = waktu_simulasi

# Ekstraksi Fitur Cerdas dari Datetime:
df['jam'] = df['waktu_transaksi'].dt.hour
df['hari_ke'] = df['waktu_transaksi'].dt.day_name()
df['is_weekend'] = df['waktu_transaksi'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0) # 1 jika Sabtu/Minggu

print("Contoh Fitur Waktu yang Telah Diekstrak:")
df[['waktu_transaksi', 'jam', 'hari_ke', 'is_weekend']].head(4)

### B. Teknik 2: Diskretisasi Nilai Kontinu (*Binning* via `pd.cut` & `pd.qcut`)
Seringkali mengelompokkan angka kontinu menjadi kategori memberikan sinyal bisnis yang lebih kuat:
- **`pd.cut`**: Membagi data ke dalam rentang interval angka yang sama lebar (misal: Tagihan Murah < $15, Sedang $15-$30, Mahal > $30).
- **`pd.qcut`**: Membagi data berdasarkan kuantil/persentase sehingga setiap kelompok memiliki jumlah baris data yang sama banyak.

In [ ]:
# Mengelompokkan total tagihan menjadi 3 kategori menggunakan pd.cut
label_kategori = ['Hemat', 'Menengah', 'Mewah']
batas_harga = [0, 15, 30, 100]

df['kategori_tagihan'] = pd.cut(df['total_bill'], bins=batas_harga, labels=label_kategori)
print("Distribusi Kategori Tagihan (pd.cut):")
print(df['kategori_tagihan'].value_counts())
df[['total_bill', 'kategori_tagihan']].head(4)

### C. Teknik 3: Pembuatan Fitur Rasio & Normalisasi Pengeluaran
Dua fitur baru yang sangat bermakna:
1. **`tip_pct`**: Persentase tip dari total tagihan (`tip / total_bill * 100`).
2. **`bill_per_person`**: Estimasi biaya makan per kepala rombongan (`total_bill / size`).

In [ ]:
df['tip_pct'] = (df['tip'] / df['total_bill']) * 100
df['bill_per_person'] = df['total_bill'] / df['size']

print("Fitur Rasio Baru:")
df[['total_bill', 'tip', 'size', 'tip_pct', 'bill_per_person']].head()

---

## 🎯 6. Seleksi Fitur & Bahaya *Curse of Dimensionality*

### Apa Itu *Curse of Dimensionality* (Bencana Dimensi)?
Ada mitos di kalangan pemula: *"Semakin banyak kolom/fitur yang kita buat, model pasti makin pintar!"*.  
**SALAH BESAR!**  
- Jika Anda menambahkan terlalu banyak fitur yang tidak relevan atau redundan, ruang data menjadi sangat renggang (*sparse*).
- Komputer membutuhkan jutaan data lebih banyak hanya untuk mengisi kekosongan ruang tersebut, model menjadi sangat lambat, dan model rentan mengalami **Overfitting** (menghafal derau/noise bukannya belajar pola asli).

### Cara Memilih Fitur Terbaik: *Feature Importance*
Kita bisa melatih model Random Forest cepat untuk melihat fitur mana yang paling berkontribusi dalam menentukan besar tip:

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Memilih fitur kandidat numerik
fitur_kandidat = ['total_bill', 'size', 'jam', 'is_weekend', 'bill_per_person']
X = df[fitur_kandidat]
y = df['tip']

# Melatih model Random Forest sederhana untuk melihat bobot pengaruh tiap fitur
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Visualisasi Skor Kepentingan Fitur (Feature Importance)
skor_fitur = pd.Series(rf.feature_importances_, index=fitur_kandidat).sort_values(ascending=True)

plt.figure(figsize=(7, 3.5))
skor_fitur.plot(kind='barh', color='darkorange')
plt.title('Skor Kepentingan Fitur Terhadap Besar Tip (Feature Importance)', fontweight='bold')
plt.xlabel('Tingkat Pengaruh (Importance Score)')
plt.show()

print("Fitur Paling Berpengaruh Teratas:\n", skor_fitur.sort_values(ascending=False))

💡 **Kesimpulan Seleksi Fitur:**  
`total_bill` dan `bill_per_person` menyumbang lebih dari 75% pengaruh terhadap besar tip! Fitur seperti `is_weekend` memiliki pengaruh sangat kecil sehingga bisa dipertimbangkan untuk dieliminasi guna meringankan model.

---

## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep EDA dan rekayasa fitur di Modul 3:

In [ ]:
# Evaluasi Pemahaman Mandiri Modul 3
evaluasi_m3 = [
    {
        'no': 1,
        'soal': 'Apa perbedaan mendasar antara analisis Univariat dan Bivariat?',
        'jawaban': 'Univariat menganalisis 1 kolom secara terpisah (seperti distribusi/skewness), sedangkan Bivariat menganalisis hubungan antar 2 kolom (seperti korelasi/tabel silang).'
    },
    {
        'no': 2,
        'soal': 'Kapan korelasi Spearman lebih dianjurkan daripada korelasi Pearson?',
        'jawaban': 'Ketika data tidak berdistribusi normal, memiliki outlier, atau hubungannya monotonik non-linear (berbasis urutan peringkat).'
    },
    {
        'no': 3,
        'soal': 'Apa bahaya utama dari Curse of Dimensionality?',
        'jawaban': 'Terlalu banyak fitur membuat ruang data menjadi renggang, komputasi menjadi sangat lambat, dan model rentan mengalami Overfitting.'
    },
    {
        'no': 4,
        'soal': 'Apa perbedaan fungsi pd.cut dan pd.qcut pada Pandas?',
        'jawaban': 'pd.cut membagi data berdasarkan batas interval nilai yang sama lebar, sedangkan pd.qcut membagi data berdasarkan kuantil/persentase data yang sama banyak.'
    }
]

for ev in evaluasi_m3:
    print(f"Soal {ev['no']}: {ev['soal']}")
    print(f"👉 Kunci: {ev['jawaban']}\n")
print("Jika Anda memahami ke-4 poin di atas, keahlian analisis data Anda sudah setara praktisi industri! 🌟")

---

## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

Ujilah keahlian rekayasa fitur Anda dengan menyelesaikan tantangan terpandu ini:

In [ ]:
# TANTANGAN MANDIRI:
# 1. Dari tabel df di atas, cari tahu hari apa ('day') yang memberikan rata-rata persentase tip ('tip_pct') tertinggi!
# 2. Buat kolom baru bernama 'rombongan_besar': bernilai 1 jika size > 3, dan bernilai 0 jika size <= 3.

# Tulis kode latihan Anda:
rata_tip_hari = df.groupby('day', observed=False)['tip_pct'].mean().sort_values(ascending=False)
df['rombongan_besar'] = df['size'].apply(lambda x: 1 if x > 3 else 0)

print("=== RATA-RATA PERSENTASE TIP BERDASARKAN HARI ===")
print(rata_tip_hari.round(2))
print(f"\nHari Paling Dermawan Memberi Tip: {rata_tip_hari.index[0]} ({rata_tip_hari.iloc[0]:.2f}%) 💵")
print("\nJumlah Rombongan Besar vs Kecil:")
print(df['rombongan_besar'].value_counts())

---

## 🎯 9. Checklist Kelulusan Modul 03 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika Anda sudah menguasai poin-poin berikut:
- [ ] Saya paham pentingnya kutipan Andrew Ng tentang rekayasa fitur (*Feature Engineering*).
- [ ] Saya bisa melakukan Analisis Univariat: menghitung skewness, kurtosis, dan histogram.
- [ ] Saya paham perbedaan korelasi linear Pearson dan korelasi peringkat Spearman.
- [ ] Saya bisa membuat tabulasi silang kategori dengan `pd.crosstab()`.
- [ ] Saya bisa menggambar matriks visual multivariat dengan `sns.pairplot` dan Heatmap korelasi.
- [ ] Saya bisa mengekstrak fitur dari kolom tanggal (`.dt.hour`, `.dt.dayofweek`, is_weekend).
- [ ] Saya bisa melakukan pengelompokan (*binning*) menggunakan `pd.cut()` dan `pd.qcut()`.
- [ ] Saya bisa merekayasa fitur rasio (`tip_pct`, `bill_per_person`).
- [ ] Saya paham bahaya *Curse of Dimensionality*.
- [ ] Saya bisa menggunakan *Feature Importance* Random Forest untuk menyeleksi fitur terbaik.

---

## 🏁 Gerbang Menuju Level 2: Algoritma Klasifikasi!

Selamat! Anda kini telah menguasai seni menganalisis data dan merekayasa fitur-fitur baru yang cerdas.

Sekarang, saatnya memasuki **inti terdalam dari Data Mining: Melatih Algoritma Machine Learning**!

Mari kita mulai dari ranah **Supervised Learning - Klasifikasi (K-NN, Naive Bayes, Decision Tree, Random Forest)**:

👉 **[Buka Modul 04: Supervised Learning I - Klasifikasi](04_Classification_Algorithms.ipynb)**

Sampai jumpa di Modul 04! Tetap konsisten dan nikmati setiap proses belajarnya! 🚀